In [1]:
import sys
import requests
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, 
    QLineEdit, QPushButton, QLabel, QComboBox, QFrame,
    QGraphicsDropShadowEffect
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal
from PyQt5.QtGui import QColor, QFont


# ----------------------------------------------------
# 1. Background Worker Thread (Prevents UI Freezing)
# ----------------------------------------------------
class WeatherWorker(QThread):
    data_fetched = pyqtSignal(dict)
    error_occurred = pyqtSignal(str)

    def __init__(self, city_name, api_key):
        super().__init__()
        self.city_name = city_name
        self.api_key = api_key

    def run(self):
        # OpenWeatherMap API Endpoint
        url = f"https://api.openweathermap.org/data/2.5/weather?q={self.city_name}&units=metric&appid={self.api_key}"
        try:
            response = requests.get(url, timeout=5)
            data = response.json()

            if response.status_code == 200:
                self.data_fetched.emit(data)
            elif response.status_code == 404:
                self.error_occurred.emit("City not found. Please check spelling.")
            elif response.status_code == 401:
                self.error_occurred.emit("Invalid API Key. Please update your key.")
            else:
                self.error_occurred.emit(f"Error: {data.get('message', 'Failed to fetch data')}")
        except requests.exceptions.RequestException:
            self.error_occurred.emit("Network error. Check your internet connection.")


# ----------------------------------------------------
# 2. Main Weather App Window
# ----------------------------------------------------
class WeatherApp(QWidget):
    # Get a free key at: https://home.openweathermap.org/users/sign_up
    API_KEY = "YOUR_OPENWEATHERMAP_API_KEY_HERE"

    PAKISTAN_CITIES = [
        "Islamabad", "Karachi", "Lahore", "Peshawar", "Quetta",
        "Rawalpindi", "Multan", "Faisalabad", "Sialkot", "Hyderabad",
        "Gujranwala", "Abbottabad", "Skardu", "Gilgit", "Gwadar"
    ]

    def __init__(self):
        super().__init__()
        self.setWindowTitle("Live Weather Pakistan")
        self.setFixedSize(420, 560)
        self.initUI()

    def initUI(self):
        # Main Layout
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(25, 25, 25, 25)
        main_layout.setSpacing(15)
        self.setLayout(main_layout)

        # --- Top Search Bar (Combo Box + Search Button) ---
        search_layout = QHBoxLayout()
        search_layout.setSpacing(10)

        self.city_input = QComboBox(self)
        self.city_input.setEditable(True)
        self.city_input.addItems(self.PAKISTAN_CITIES)
        self.city_input.lineEdit().setPlaceholderText("Select or type a city...")

        self.get_weather_btn = QPushButton("Search", self)
        self.get_weather_btn.setCursor(Qt.PointingHandCursor)
        self.get_weather_btn.clicked.connect(self.fetch_weather)

        search_layout.addWidget(self.city_input, stretch=3)
        search_layout.addWidget(self.get_weather_btn, stretch=1)
        main_layout.addLayout(search_layout)

        # --- Main Weather Display Card ---
        self.card = QFrame(self)
        self.card.setObjectName("WeatherCard")
        card_layout = QVBoxLayout(self.card)
        card_layout.setAlignment(Qt.AlignCenter)
        card_layout.setSpacing(8)

        # Weather Emoji Display
        self.emoji_label = QLabel("☀️", self)
        self.emoji_label.setAlignment(Qt.AlignCenter)
        self.emoji_label.setObjectName("EmojiLabel")

        # Temperature Label
        self.temp_label = QLabel("--°C", self)
        self.temp_label.setAlignment(Qt.AlignCenter)
        self.temp_label.setObjectName("TempLabel")

        # Description Label
        self.description_label = QLabel("Select a city to view weather", self)
        self.description_label.setAlignment(Qt.AlignCenter)
        self.description_label.setObjectName("DescLabel")
        self.description_label.setWordWrap(True)

        card_layout.addWidget(self.emoji_label)
        card_layout.addWidget(self.temp_label)
        card_layout.addWidget(self.description_label)
        main_layout.addWidget(self.card)

        # --- Extra Weather Metrics Card (Humidity, Wind, Feels Like) ---
        metrics_frame = QFrame(self)
        metrics_frame.setObjectName("MetricsFrame")
        metrics_layout = QHBoxLayout(metrics_frame)
        metrics_layout.setContentsMargins(15, 12, 15, 12)

        self.feels_label = QLabel("Feels: --°C", self)
        self.humidity_label = QLabel("Humidity: --%", self)
        self.wind_label = QLabel("Wind: -- km/h", self)

        for label in (self.feels_label, self.humidity_label, self.wind_label):
            label.setAlignment(Qt.AlignCenter)
            label.setObjectName("MetricLabel")
            metrics_layout.addWidget(label)

        main_layout.addWidget(metrics_frame)

        # Connect ENTER key in city input to trigger search
        self.city_input.lineEdit().returnPressed.connect(self.fetch_weather)

        # Apply Modern Dark Stylesheet
        self.apply_stylesheet()

        # Fetch weather for default city (Islamabad) on startup
        self.fetch_weather()

    def fetch_weather(self):
        city = self.city_input.currentText().strip()
        if not city:
            self.description_label.setText("Please enter or select a city.")
            return

        # UI Loading State
        self.get_weather_btn.setEnabled(False)
        self.get_weather_btn.setText("...")
        self.description_label.setText(f"Fetching data for {city}...")

        # Spawn Background Worker
        self.worker = WeatherWorker(city, self.API_KEY)
        self.worker.data_fetched.connect(self.display_weather)
        self.worker.error_occurred.connect(self.display_error)
        self.worker.finished.connect(self.reset_button_state)
        self.worker.start()

    def reset_button_state(self):
        self.get_weather_btn.setEnabled(True)
        self.get_weather_btn.setText("Search")

    def display_weather(self, data):
        try:
            temp = round(data["main"]["temp"])
            feels_like = round(data["main"]["feels_like"])
            humidity = data["main"]["humidity"]
            wind_speed = round(data["wind"]["speed"] * 3.6)  # m/s to km/h
            weather_id = data["weather"][0]["id"]
            description = data["weather"][0]["description"].title()
            city_name = data["name"]
            country = data["sys"].get("country", "")

            # Update Labels
            self.temp_label.setText(f"{temp}°C")
            self.description_label.setText(f"{description}\nin {city_name}, {country}")
            self.emoji_label.setText(self.get_weather_emoji(weather_id))
            
            self.feels_label.setText(f"Feels: {feels_like}°C")
            self.humidity_label.setText(f"Humidity: {humidity}%")
            self.wind_label.setText(f"Wind: {wind_speed} km/h")

        except KeyError:
            self.display_error("Error parsing weather data.")

    def display_error(self, message):
        self.emoji_label.setText("⚠️")
        self.temp_label.setText("--°C")
        self.description_label.setText(message)
        self.feels_label.setText("Feels: --°C")
        self.humidity_label.setText("Humidity: --%")
        self.wind_label.setText("Wind: -- km/h")

    @staticmethod
    def get_weather_emoji(weather_id):
        # OpenWeatherMap condition codes mapping
        if 200 <= weather_id <= 232:
            return "⛈️"  # Thunderstorm
        elif 300 <= weather_id <= 321:
            return "🌧️"  # Drizzle
        elif 500 <= weather_id <= 531:
            return "🌧️"  # Rain
        elif 600 <= weather_id <= 622:
            return "❄️"  # Snow
        elif 701 <= weather_id <= 781:
            return "🌫️"  # Atmosphere (Haze / Fog / Dust)
        elif weather_id == 800:
            return "☀️"  # Clear / Sunny
        elif 801 <= weather_id <= 804:
            return "☁️"  # Clouds
        return "❓"

    def apply_stylesheet(self):
        self.setStyleSheet("""
            QWidget {
                background-color: #0F172A;
                font-family: 'Segoe UI', Arial, sans-serif;
            }

            /* --- City Dropdown / Input --- */
            QComboBox {
                background-color: #1E293B;
                color: #F8FAFC;
                border: 1px solid #334155;
                border-radius: 8px;
                padding: 8px 12px;
                font-size: 15px;
            }
            QComboBox:focus {
                border-color: #38BDF8;
            }
            QComboBox QAbstractItemView {
                background-color: #1E293B;
                color: #F8FAFC;
                selection-background-color: #0284C7;
                border: 1px solid #334155;
            }

            /* --- Search Button --- */
            QPushButton {
                background-color: #0284C7;
                color: #FFFFFF;
                font-size: 15px;
                font-weight: bold;
                border: none;
                border-radius: 8px;
                padding: 8px 16px;
            }
            QPushButton:hover {
                background-color: #0369A1;
            }
            QPushButton:pressed {
                background-color: #075985;
            }

            /* --- Weather Card --- */
            QFrame#WeatherCard {
                background-color: #1E293B;
                border-radius: 16px;
                border: 1px solid #334155;
            }

            QLabel#EmojiLabel {
                font-size: 90px;
                background: transparent;
            }

            QLabel#TempLabel {
                font-size: 56px;
                font-weight: bold;
                color: #F8FAFC;
                background: transparent;
            }

            QLabel#DescLabel {
                font-size: 18px;
                color: #94A3B8;
                font-weight: 500;
                background: transparent;
            }

            /* --- Metrics Card --- */
            QFrame#MetricsFrame {
                background-color: #1E293B;
                border-radius: 12px;
                border: 1px solid #334155;
            }

            QLabel#MetricLabel {
                font-size: 13px;
                color: #38BDF8;
                font-weight: 600;
                background: transparent;
            }
        """)


# ----------------------------------------------------
# 3. App Execution
# ----------------------------------------------------
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = WeatherApp()
    window.show()
    sys.exit(app.exec_())

SystemExit: 0

c:\Users\pts\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import sys
import requests
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, 
    QComboBox, QPushButton, QLabel, QFrame
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal

# ----------------------------------------------------
# 1. Background Worker Thread (No API Key Required!)
# ----------------------------------------------------
class WeatherWorker(QThread):
    data_fetched = pyqtSignal(dict)
    error_occurred = pyqtSignal(str)

    def __init__(self, city_name):
        super().__init__()
        self.city_name = city_name

    def run(self):
        try:
            # Step 1: Geocoding (Convert city name to Latitude/Longitude)
            geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={self.city_name}&count=1&language=en&format=json"
            geo_res = requests.get(geo_url, timeout=5)
            geo_data = geo_res.json()

            if not geo_data.get("results"):
                self.error_occurred.emit("City not found. Check spelling.")
                return

            location = geo_data["results"][0]
            lat = location["latitude"]
            lon = location["longitude"]
            city_display = location["name"]
            country = location.get("country", "")

            # Step 2: Fetch Live Weather using Lat/Lon
            weather_url = (
                f"https://api.open-meteo.com/v1/forecast?"
                f"latitude={lat}&longitude={lon}&current="
                f"temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m"
            )
            w_res = requests.get(weather_url, timeout=5)
            w_data = w_res.json()

            current = w_data["current"]
            
            # Bundle data for main window UI
            result = {
                "city": city_display,
                "country": country,
                "temp": round(current["temperature_2m"]),
                "feels_like": round(current["apparent_temperature"]),
                "humidity": current["relative_humidity_2m"],
                "wind": round(current["wind_speed_10m"]),
                "code": current["weather_code"]
            }
            self.data_fetched.emit(result)

        except requests.exceptions.RequestException:
            self.error_occurred.emit("Network error. Check your internet connection.")
        except Exception as e:
            self.error_occurred.emit(f"Failed to fetch data: {str(e)}")


# ----------------------------------------------------
# 2. Main Weather App Window
# ----------------------------------------------------
class WeatherApp(QWidget):
    PAKISTAN_CITIES = [
        "Peshawar", "Islamabad", "Karachi", "Lahore", "Quetta",
        "Rawalpindi", "Multan", "Faisalabad", "Sialkot", "Hyderabad",
        "Gujranwala", "Abbottabad", "Skardu", "Gilgit", "Gwadar"
    ]

    def __init__(self):
        super().__init__()
        self.setWindowTitle("Live Weather Pakistan")
        self.setFixedSize(420, 560)
        self.initUI()

    def initUI(self):
        # Main Layout
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(25, 25, 25, 25)
        main_layout.setSpacing(15)
        self.setLayout(main_layout)

        # --- Search Layout ---
        search_layout = QHBoxLayout()
        search_layout.setSpacing(10)

        self.city_input = QComboBox(self)
        self.city_input.setEditable(True)
        self.city_input.addItems(self.PAKISTAN_CITIES)
        self.city_input.lineEdit().setPlaceholderText("Select or type city...")

        self.get_weather_btn = QPushButton("Search", self)
        self.get_weather_btn.setCursor(Qt.PointingHandCursor)
        self.get_weather_btn.clicked.connect(self.fetch_weather)

        search_layout.addWidget(self.city_input, stretch=3)
        search_layout.addWidget(self.get_weather_btn, stretch=1)
        main_layout.addLayout(search_layout)

        # --- Weather Display Card ---
        self.card = QFrame(self)
        self.card.setObjectName("WeatherCard")
        card_layout = QVBoxLayout(self.card)
        card_layout.setAlignment(Qt.AlignCenter)
        card_layout.setSpacing(8)

        self.emoji_label = QLabel("☀️", self)
        self.emoji_label.setAlignment(Qt.AlignCenter)
        self.emoji_label.setObjectName("EmojiLabel")

        self.temp_label = QLabel("--°C", self)
        self.temp_label.setAlignment(Qt.AlignCenter)
        self.temp_label.setObjectName("TempLabel")

        self.description_label = QLabel("Select a city to view weather", self)
        self.description_label.setAlignment(Qt.AlignCenter)
        self.description_label.setObjectName("DescLabel")
        self.description_label.setWordWrap(True)

        card_layout.addWidget(self.emoji_label)
        card_layout.addWidget(self.temp_label)
        card_layout.addWidget(self.description_label)
        main_layout.addWidget(self.card)

        # --- Metrics Card ---
        metrics_frame = QFrame(self)
        metrics_frame.setObjectName("MetricsFrame")
        metrics_layout = QHBoxLayout(metrics_frame)
        metrics_layout.setContentsMargins(15, 12, 15, 12)

        self.feels_label = QLabel("Feels: --°C", self)
        self.humidity_label = QLabel("Humidity: --%", self)
        self.wind_label = QLabel("Wind: -- km/h", self)

        for label in (self.feels_label, self.humidity_label, self.wind_label):
            label.setAlignment(Qt.AlignCenter)
            label.setObjectName("MetricLabel")
            metrics_layout.addWidget(label)

        main_layout.addWidget(metrics_frame)

        self.city_input.lineEdit().returnPressed.connect(self.fetch_weather)
        self.apply_stylesheet()

        # Load Peshawar by default on startup
        self.fetch_weather()

    def fetch_weather(self):
        city = self.city_input.currentText().strip()
        if not city:
            self.description_label.setText("Please enter or select a city.")
            return

        self.get_weather_btn.setEnabled(False)
        self.get_weather_btn.setText("...")
        self.description_label.setText(f"Fetching data for {city}...")

        self.worker = WeatherWorker(city)
        self.worker.data_fetched.connect(self.display_weather)
        self.worker.error_occurred.connect(self.display_error)
        self.worker.finished.connect(self.reset_button_state)
        self.worker.start()

    def reset_button_state(self):
        self.get_weather_btn.setEnabled(True)
        self.get_weather_btn.setText("Search")

    def display_weather(self, data):
        emoji, condition_text = self.get_wmo_condition(data["code"])

        self.temp_label.setText(f"{data['temp']}°C")
        self.description_label.setText(f"{condition_text}\nin {data['city']}, {data['country']}")
        self.emoji_label.setText(emoji)

        self.feels_label.setText(f"Feels: {data['feels_like']}°C")
        self.humidity_label.setText(f"Humidity: {data['humidity']}%")
        self.wind_label.setText(f"Wind: {data['wind']} km/h")

    def display_error(self, message):
        self.emoji_label.setText("⚠️")
        self.temp_label.setText("--°C")
        self.description_label.setText(message)
        self.feels_label.setText("Feels: --°C")
        self.humidity_label.setText("Humidity: --%")
        self.wind_label.setText("Wind: -- km/h")

    @staticmethod
    def get_wmo_condition(code):
        # Maps WMO Weather Codes to Emoji & Description
        mapping = {
            0: ("☀️", "Clear Sky"),
            1: ("🌤️", "Mainly Clear"),
            2: ("⛅", "Partly Cloudy"),
            3: ("☁️", "Overcast"),
            45: ("🌫️", "Foggy"),
            48: ("🌫️", "Depositing Rime Fog"),
            51: ("🌧️", "Light Drizzle"),
            53: ("🌧️", "Moderate Drizzle"),
            55: ("🌧️", "Dense Drizzle"),
            61: ("🌧️", "Slight Rain"),
            63: ("🌧️", "Moderate Rain"),
            65: ("🌧️", "Heavy Rain"),
            71: ("❄️", "Slight Snow"),
            73: ("❄️", "Moderate Snow"),
            75: ("❄️", "Heavy Snow"),
            80: ("🌦️", "Slight Rain Showers"),
            81: ("🌦️", "Moderate Rain Showers"),
            82: ("🌧️", "Violent Rain Showers"),
            95: ("⛈️", "Thunderstorm"),
            96: ("⛈️", "Thunderstorm with Hail"),
            99: ("⛈️", "Heavy Thunderstorm")
        }
        return mapping.get(code, ("🌍", "Weather Update"))

    def apply_stylesheet(self):
        self.setStyleSheet("""
            QWidget {
                background-color: #0F172A;
                font-family: 'Segoe UI', Arial, sans-serif;
            }

            QComboBox {
                background-color: #1E293B;
                color: #F8FAFC;
                border: 1px solid #334155;
                border-radius: 8px;
                padding: 8px 12px;
                font-size: 15px;
            }
            QComboBox:focus {
                border-color: #38BDF8;
            }
            QComboBox QAbstractItemView {
                background-color: #1E293B;
                color: #F8FAFC;
                selection-background-color: #0284C7;
                border: 1px solid #334155;
            }

            QPushButton {
                background-color: #0284C7;
                color: #FFFFFF;
                font-size: 15px;
                font-weight: bold;
                border: none;
                border-radius: 8px;
                padding: 8px 16px;
            }
            QPushButton:hover {
                background-color: #0369A1;
            }
            QPushButton:pressed {
                background-color: #075985;
            }

            QFrame#WeatherCard {
                background-color: #1E293B;
                border-radius: 16px;
                border: 1px solid #334155;
            }

            QLabel#EmojiLabel {
                font-size: 90px;
                background: transparent;
            }

            QLabel#TempLabel {
                font-size: 56px;
                font-weight: bold;
                color: #F8FAFC;
                background: transparent;
            }

            QLabel#DescLabel {
                font-size: 18px;
                color: #94A3B8;
                font-weight: 500;
                background: transparent;
            }

            QFrame#MetricsFrame {
                background-color: #1E293B;
                border-radius: 12px;
                border: 1px solid #334155;
            }

            QLabel#MetricLabel {
                font-size: 13px;
                color: #38BDF8;
                font-weight: 600;
                background: transparent;
            }
        """)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = WeatherApp()
    window.show()
    sys.exit(app.exec_())

SystemExit: 0

c:\Users\pts\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
